# 02. Modelado Predictivo Supervisado
**Asignatura:** Programación para la Ciencia de Datos (SCY1101)  
**Integrantes:** Kimberly Bobadilla, Martina Ortiz, Sebastián Parada  

### Objetivo:
Entrenar dos modelos de aprendizaje supervisado (Árbol de Decisión y Random Forest) para clasificar y predecir el `Rango_precio` de los productos en la región de Arica y Parinacota, utilizando las variables temporales, sectoriales y la segmentación no supervisada (clústeres) generada en la etapa anterior.

In [11]:
# 1. Configuración de rutas si trabajas en Google Colab
import os
import pandas as pd
import numpy as np

# Verificar si estamos en entorno Colab para montar/clonar
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('SCY1101_EP2_Grupo11'):
        !git clone https://github.com/jxmartii/SCY1101_EP2_Grupo11.git
    os.chdir('/content/SCY1101_EP2_Grupo11/notebooks')

# 2. Cargar el dataset que guardamos con la columna de clústeres
df_modelo = pd.read_csv('../data/precio_consumidor_arica_clusterizado.csv')
print(f"Dataset cargado con éxito. Dimensiones: {df_modelo.shape}")
df_modelo.head(3)

Dataset cargado con éxito. Dimensiones: (12417, 14)


,Mes,Semana,Fecha inicio,Fecha termino,Sector,Tipo de punto monitoreo,Grupo,Producto,Unidad,Precio minimo,Precio maximo,Precio promedio,Rango_precio,Cluster
0,1.0,1.0,2024-12-30,2025-01-03,Arica,Carnicería,Carne bovina,Abastero,$/kilo,8499.0,8799.0,8599.0,300.0,2.0
1,1.0,1.0,2024-12-30,2025-01-03,Arica,Carnicería,Carne bovina,Asado Carnicero,$/kilo,7999.0,7999.0,7999.0,0.0,2.0
2,1.0,1.0,2024-12-30,2025-01-03,Arica,Carnicería,Carne bovina,Estomaguillo (Tapabarriga),$/kilo,7999.0,7999.0,7999.0,0.0,2.0


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Eliminar nulos en el precio
df_modelo_limpio = df_modelo.dropna(subset=['Rango_precio']).copy()

# 2. Convertir los números continuos a categorías reales (Bajo, Medio, Alto) usando cuantiles
# Esto soluciona de raíz el error al agrupar los datos en 3 tramos equilibrados
df_modelo_limpio['Rango_precio_categoria'] = pd.qcut(
    df_modelo_limpio['Rango_precio'], 
    q=3, 
    labels=['Bajo', 'Medio', 'Alto']
)

# 3. Seleccionar columnas predictoras (X) y la nueva variable objetivo categórica (y)
columnas_cluster = ['Mes', 'Semana', 'Sector', 'Grupo', 'Cluster']
X_raw = df_modelo_limpio[columnas_cluster]
y = df_modelo_limpio['Rango_precio_categoria']

# 4. Convertir variables categóricas a números con One-Hot Encoding
X_encoded = pd.get_dummies(X_raw, columns=['Sector', 'Grupo'], drop_first=True)

# 5. Dividir el dataset en Entrenamiento (80%) y Prueba (20%) de forma balanceada
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print("--- PARTICIÓN DE DATOS COMPLETADA CON ÉXITO ---")
print(f"Dimensiones de entrenamiento (X_train): {X_train.shape}")
print(f"Dimensiones de prueba (X_test): {X_test.shape}")
print("\nDistribución de las nuevas categorías creadas:")
print(y.value_counts())

--- PARTICIÓN DE DATOS COMPLETADA CON ÉXITO ---
Dimensiones de entrenamiento (X_train): (8620, 19)
Dimensiones de prueba (X_test): (2156, 19)

Distribución de las nuevas categorías creadas:
Rango_precio_categoria
Medio    3637
Bajo     3592
Alto     3547
Name: count, dtype: int64


In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Instanciar el primer modelo (Árbol de Decisión) con valores por defecto
modelo_tree = DecisionTreeClassifier(random_state=42)

# 2. Entrenar el modelo
modelo_tree.fit(X_train, y_train)

# 3. Realizar predicciones iniciales
preds_tree = modelo_tree.predict(X_test)

# 4. Mostrar rendimiento inicial preliminar
acc_tree = accuracy_score(y_test, preds_tree)
print("=== RENDIMIENTO INICIAL: ÁRBOL DE DECISIÓN ===")
print(f"Accuracy Inicial: {acc_tree:.4f}\n")
print(classification_report(y_test, preds_tree))

=== RENDIMIENTO INICIAL: ÁRBOL DE DECISIÓN ===
Accuracy Inicial: 0.5626

              precision    recall  f1-score   support

        Alto       0.69      0.61      0.65       709
        Bajo       0.48      0.47      0.48       719
       Medio       0.53      0.61      0.57       728

    accuracy                           0.56      2156
   macro avg       0.57      0.56      0.56      2156
weighted avg       0.57      0.56      0.56      2156



In [14]:
from sklearn.ensemble import RandomForestClassifier

# 1. Instanciar el segundo modelo (Random Forest) para contrastar el rendimiento
modelo_rf = RandomForestClassifier(random_state=42, n_estimators=100)

# 2. Entrenar el modelo
modelo_rf.fit(X_train, y_train)

# 3. Realizar predicciones iniciales
preds_rf = modelo_rf.predict(X_test)

# 4. Mostrar rendimiento inicial preliminar
acc_rf = accuracy_score(y_test, preds_rf)
print("=== RENDIMIENTO INICIAL: RANDOM FOREST ===")
print(f"Accuracy Inicial: {acc_rf:.4f}\n")
print(classification_report(y_test, preds_rf))

=== RENDIMIENTO INICIAL: RANDOM FOREST ===
Accuracy Inicial: 0.5575

              precision    recall  f1-score   support

        Alto       0.71      0.58      0.64       709
        Bajo       0.48      0.46      0.47       719
       Medio       0.52      0.63      0.57       728

    accuracy                           0.56      2156
   macro avg       0.57      0.56      0.56      2156
weighted avg       0.57      0.56      0.56      2156



In [16]:
import joblib

# Crear carpeta de modelos si no existe en la raíz
os.makedirs('../models', exist_ok=True)

# Guardar los objetos del modelo entrenados y los datos divididos para usarlos en el Notebook 03
joblib.dump(modelo_tree, '../models/baseline_decision_tree.pkl')
joblib.dump(modelo_rf, '../models/baseline_random_forest.pkl')

# Guardar conjuntos de datos temporales para mantener consistencia entre cuadernos
joblib.dump((X_train, X_test, y_train, y_test), '../models/datos_entrenamiento.pkl')

print("Modelos base y particiones guardados exitosamente en la carpeta /models")

Modelos base y particiones guardados exitosamente en la carpeta /models
